# Handle and DID Resolution

This notebook builds handle and DID resolution, core operations in ATProto identity.
ATProto uses two identifiers: human-readable handles (like `alice.bsky.social`) 
and cryptographic DIDs (like `did:plc:abc123`). Resolution maps between them.

**What you'll learn:**
- Handle → DID resolution
- DID → DID document resolution  
- Two-tier identity resolution (handle → DID → document)

**Estimated Time:** 15-20 minutes

## Step 1: Handle Resolver

The handle resolver maps handles to DIDs. In a real ATProto implementation,
handles are resolved via DNS TXT records or HTTP well-known endpoints.
Here we use an in-memory dictionary for simplicity.

In [ ]:
@interface HandleResolver : NSObject
@property (nonatomic, strong) NSMutableDictionary *handleToDid;
- (void)registerHandle:(NSString *)handle did:(NSString *)did;
- (NSString *)resolveHandle:(NSString *)handle;
- (BOOL)isValidHandleFormat:(NSString *)handle;
@end

@implementation HandleResolver
- (instancetype)init {
    self = [super init];
    if (self) { _handleToDid = [NSMutableDictionary dictionary]; }
    return self;
}
- (void)registerHandle:(NSString *)handle did:(NSString *)did {
    [_handleToDid setObject:did forKey:handle];
    NSLog(@"Registered: %@ -> %@", handle, did);
}
- (NSString *)resolveHandle:(NSString *)handle {
    NSString *did = [_handleToDid objectForKey:handle];
    if (did != nil) {
        NSLog(@"Resolved %@ -> %@", handle, did);
    } else {
        NSLog(@"Handle not found: %@", handle);
    }
    return did;
}
- (BOOL)isValidHandleFormat:(NSString *)handle {
    if (handle == nil || [handle length] < 3) return NO;
    if (![handle containsString:@"."]) return NO;
    if ([handle containsString:@" "]) return NO;
    return YES;
}
@end

HandleResolver *hr = [[HandleResolver alloc] init];
[hr registerHandle:@"alice.bsky.social" did:@"did:plc:abc123"];
[hr registerHandle:@"bob.example.com" did:@"did:plc:def456"];

NSLog(@"Alice: %@", [hr resolveHandle:@"alice.bsky.social"]);
NSLog(@"Unknown: %@", [hr resolveHandle:@"unknown.bsky.social"]);
NSLog(@"Valid format: %d", [hr isValidHandleFormat:@"alice.bsky.social"]);
NSLog(@"Invalid: %d", [hr isValidHandleFormat:@"nodot"]);

## Step 2: DID Resolver

The DID resolver maps DIDs to DID documents. A DID document contains
`alsoKnownAs` (associated handles), `verificationMethod` (public keys), and `service` endpoints.
In practice, DID documents are fetched from PLC directories or web DID endpoints.

In [ ]:
@interface DIDResolver : NSObject
@property (nonatomic, strong) NSMutableDictionary *didToDocument;
- (void)registerDid:(NSString *)did document:(NSDictionary *)document;
- (NSDictionary *)resolveDid:(NSString *)did;
@end

@implementation DIDResolver
- (instancetype)init {
    self = [super init];
    if (self) { _didToDocument = [NSMutableDictionary dictionary]; }
    return self;
}
- (void)registerDid:(NSString *)did document:(NSDictionary *)document {
    [_didToDocument setObject:document forKey:did];
    NSLog(@"Registered DID: %@", did);
}
- (NSDictionary *)resolveDid:(NSString *)did {
    NSDictionary *doc = [_didToDocument objectForKey:did];
    if (doc != nil) {
        NSLog(@"Resolved DID: %@", did);
    } else {
        NSLog(@"DID not found: %@", did);
    }
    return doc;
}
@end

DIDResolver *dr = [[DIDResolver alloc] init];
[dr registerDid:@"did:plc:abc123" document:@{
    @"alsoKnownAs": @[@"at://alice.bsky.social"],
    @"verificationMethod": @[@{@"id": @"#atproto", @"type": @"Multikey"}],
    @"service": @[@{@"id": @"#atproto_pds", @"type": @"AtprotoPersonalDataServer"}]
}];

NSDictionary *doc = [dr resolveDid:@"did:plc:abc123"];
NSLog(@"alsoKnownAs: %@", [doc objectForKey:@"alsoKnownAs"]);

## Step 3: Identity Service

The identity service chains both resolvers: handle → DID → document.
This two-tier resolution pattern is fundamental to ATProto identity.

In [ ]:
@interface IdentityService : NSObject
@property (nonatomic, strong) HandleResolver *handleResolver;
@property (nonatomic, strong) DIDResolver *didResolver;
- (NSDictionary *)resolveHandle:(NSString *)handle;
- (NSDictionary *)resolveDid:(NSString *)did;
@end

@implementation IdentityService
- (instancetype)init {
    self = [super init];
    if (self) {
        _handleResolver = [[HandleResolver alloc] init];
        _didResolver = [[DIDResolver alloc] init];
    }
    return self;
}
- (NSDictionary *)resolveHandle:(NSString *)handle {
    // Step 1: handle → DID
    NSString *did = [_handleResolver resolveHandle:handle];
    if (did == nil) return nil;
    // Step 2: DID → document
    return [_didResolver resolveDid:did];
}
- (NSDictionary *)resolveDid:(NSString *)did {
    return [_didResolver resolveDid:did];
}
@end

// Wire it up
IdentityService *idSvc = [[IdentityService alloc] init];
[idSvc.handleResolver registerHandle:@"alice.bsky.social" did:@"did:plc:abc123"];
[idSvc.didResolver registerDid:@"did:plc:abc123" document:@{
    @"alsoKnownAs": @[@"at://alice.bsky.social"],
    @"verificationMethod": @[@{@"id": @"#atproto", @"type": @"Multikey"}],
    @"service": @[@{@"id": @"#atproto_pds", @"type": @"AtprotoPersonalDataServer"}]
}];

// Resolve handle → DID → document
NSDictionary *doc = [idSvc resolveHandle:@"alice.bsky.social"];
NSLog(@"Document: %@", doc);

// Resolve DID directly
NSDictionary *doc2 = [idSvc resolveDid:@"did:plc:abc123"];
NSLog(@"Same doc: %d", [doc isEqualToDictionary:doc2]);

## Exercise

Add `updateHandle:forDid:` to `IdentityService` that updates the handle→DID mapping
and the DID document's `alsoKnownAs` array. Handle updates are a key operation in ATProto
where users can change their human-readable identifier.

Hint: remove the old handle→DID entry, add the new one, and update
the DID document's `alsoKnownAs` array.

In [ ]:
// Exercise: implement updateHandle:forDid:
// Your code here...

## Solution

In [ ]:
// Solution: updateHandle removes old mapping, adds new one, updates alsoKnownAs
@interface IdentityService2 : NSObject
@property (nonatomic, strong) HandleResolver *handleResolver;
@property (nonatomic, strong) DIDResolver *didResolver;
- (void)updateHandle:(NSString *)newHandle forDid:(NSString *)did;
@end

@implementation IdentityService2
- (instancetype)init {
    self = [super init];
    if (self) {
        _handleResolver = [[HandleResolver alloc] init];
        _didResolver = [[DIDResolver alloc] init];
    }
    return self;
}
- (void)updateHandle:(NSString *)newHandle forDid:(NSString *)did {
    // Remove old handle→DID entries for this DID
    NSMutableArray *oldHandles = [NSMutableArray array];
    for (NSString *handle in _handleResolver.handleToDid) {
        if ([[_handleResolver.handleToDid objectForKey:handle] isEqualToString:did]) {
            [oldHandles addObject:handle];
        }
    }
    for (int i = 0; i < [oldHandles count]; i++) {
        [_handleResolver.handleToDid removeObjectForKey:[oldHandles objectAtIndex:i]];
    }
    // Add new mapping
    [_handleResolver registerHandle:newHandle did:did];
    // Update DID document's alsoKnownAs
    NSDictionary *doc = [_didResolver resolveDid:did];
    if (doc != nil) {
        NSMutableDictionary *mutableDoc = [NSMutableDictionary dictionaryWithDictionary:doc];
        [mutableDoc setObject:@[[NSString stringWithFormat:@"at://%@", newHandle]] forKey:@"alsoKnownAs"];
        [_didResolver.didToDocument setObject:mutableDoc forKey:did];
    }
    NSLog(@"Updated handle for %@ to %@", did, newHandle);
}
@end

// Test it
IdentityService2 *id2 = [[IdentityService2 alloc] init];
[id2.handleResolver registerHandle:@"alice.bsky.social" did:@"did:plc:abc"];
[id2.didResolver registerDid:@"did:plc:abc" document:@{
    @"alsoKnownAs": @[@"at://alice.bsky.social"],
    @"verificationMethod": @[]
}];

[id2 updateHandle:@"alice.new-domain.com" forDid:@"did:plc:abc"];
NSLog(@"New handle: %@", [id2.handleResolver resolveHandle:@"alice.new-domain.com"]);
NSLog(@"Old handle: %@", [id2.handleResolver resolveHandle:@"alice.bsky.social"]);

## Next Steps

You've now seen how ATProto handles identity. Next:
- Learn about **AT URIs** — how to address individual records using `at://` scheme
- Build a **Record Store** — how records are organized and retrieved
- Explore **Firehose events** — how changes to identity and records propagate across the network